# Landsat preprocessing

In this notebook we preprocess the Landsat imagery.

Landsat images were downloaded from Google Earth Engine, where it was processed with LandTrendr to produce semiannual (summer and winter) best-available-pixel composites.

In [1]:
import glob
import pathlib
import re
import remote_sensing_processor as rsp

In [2]:
landsats = glob.glob(r'G:/Work/hmao-landsat/**/**/*.tif')

## Calculating the normalization parameters

Here we pre-calculate normalization parameters (30 and 70 percentiles). We calculate these means of these parameters for each band for each season to avoid bias connected with overfitting the parameters to the specific image.

In [3]:
norm_params = {
    "summer": {
        "B1": {"p1": 5.495841828788199, "p2": 5.7804695222435925},
        "B2": {"p1": 6.066084943166593, "p2": 6.332691262408001},
        "B3": {"p1": 5.7302664896337, "p2": 6.238054926802472},
        "B4": {"p1": 7.672398590460056, "p2": 7.915162761036942},
        "B5": {"p1": 7.024796962738037, "p2": 7.328445818366074},
        "B7": {"p1": 6.220502074171857, "p2": 6.611923031690644},
    },
    "winter": {
        "B1": {"p1": 7.885436511621243, "p2": 8.892835826408572},
        "B2": {"p1": 7.8513555177828165, "p2": 8.893602859683153},
        "B3": {"p1": 7.786362206063619, "p2": 8.903115202740926},
        "B4": {"p1": 8.113724452693289, "p2": 8.870667224977074},
        "B5": {"p1": 6.593629127595483, "p2": 7.0251839800578795},
        "B7": {"p1": 6.349962304278118, "p2": 6.888074828357231},
    },
}

In [4]:
seasons = ["summer", "winter"]
bands = ["B1", "B2", "B3", "B4", "B5", "B7"]
percentiles = [30, 70]

for season in seasons:
    for band in bands:
        band_landsats = list(filter(lambda x: season in x and band in x, landsats))
        if norm_params[season][band]["p1"] == 0:
            pcs = rsp.get_normalization_params.dynamic_world(band_landsats, percentiles=percentiles, nodata=-9999)
            norm_params[season][band]["p1"] = pcs[percentiles[0]]
            norm_params[season][band]["p2"] = pcs[percentiles[1]]
        print(season, band, "p1: " + str(norm_params[season][band]["p1"]), "p2: " + str(norm_params[season][band]["p2"]))

summer B1 p1: 5.495841828788199 p2: 5.7804695222435925
summer B2 p1: 6.066084943166593 p2: 6.332691262408001
summer B3 p1: 5.7302664896337 p2: 6.238054926802472
summer B4 p1: 7.672398590460056 p2: 7.915162761036942
summer B5 p1: 7.024796962738037 p2: 7.328445818366074
summer B7 p1: 6.220502074171857 p2: 6.611923031690644
winter B1 p1: 7.885436511621243 p2: 8.892835826408572
winter B2 p1: 7.8513555177828165 p2: 8.893602859683153
winter B3 p1: 7.786362206063619 p2: 8.903115202740926
winter B4 p1: 8.113724452693289 p2: 8.870667224977074
winter B5 p1: 6.593629127595483 p2: 7.0251839800578795
winter B7 p1: 6.349962304278118 p2: 6.888074828357231


## Preprocessing

Then we normalize the data and fill the gaps (if there are any).

In [6]:
for landsat in landsats:
    in_file = landsat
    out_file = landsat.replace('hmao-landsat', 'hmao-landsat-processed')
    clip = r'G:/Work/hmao-landsat-processed/clipper.gpkg'
    if not pathlib.Path(out_file).exists():
        season = "summer" if "summer" in in_file else "winter"
        band = re.findall(r"B\d", in_file)[0]
        percentile1 = norm_params[season][band]["p1"]
        percentile2 = norm_params[season][band]["p2"]
        rsp.normalize.dynamic_world(in_file, percentile1=percentile1, percentile2=percentile2, output_path=out_file, nodata=-9999)
        rsp.process(out_file, fill_nodata=True, fill_distance=250, clip=clip)
    print(in_file)

G:/Work/hmao-landsat\1984\summer\hmao-B1-1984.tif
G:/Work/hmao-landsat\1984\summer\hmao-B2-1984.tif
G:/Work/hmao-landsat\1984\summer\hmao-B3-1984.tif
G:/Work/hmao-landsat\1984\summer\hmao-B4-1984.tif
G:/Work/hmao-landsat\1984\summer\hmao-B5-1984.tif
G:/Work/hmao-landsat\1984\summer\hmao-B7-1984.tif
G:/Work/hmao-landsat\1984\winter\hmao-B1-1984.tif
G:/Work/hmao-landsat\1984\winter\hmao-B2-1984.tif
G:/Work/hmao-landsat\1984\winter\hmao-B3-1984.tif
G:/Work/hmao-landsat\1984\winter\hmao-B4-1984.tif
G:/Work/hmao-landsat\1984\winter\hmao-B5-1984.tif
G:/Work/hmao-landsat\1984\winter\hmao-B7-1984.tif
G:/Work/hmao-landsat\1985\summer\hmao-B1-1985.tif
G:/Work/hmao-landsat\1985\summer\hmao-B2-1985.tif
G:/Work/hmao-landsat\1985\summer\hmao-B3-1985.tif
G:/Work/hmao-landsat\1985\summer\hmao-B4-1985.tif
G:/Work/hmao-landsat\1985\summer\hmao-B5-1985.tif
G:/Work/hmao-landsat\1985\summer\hmao-B7-1985.tif
G:/Work/hmao-landsat\1985\winter\hmao-B1-1985.tif
G:/Work/hmao-landsat\1985\winter\hmao-B2-1985.tif
